In [ ]:
# Step 0.0
import ultralytics
ultralytics.checks()   # verify installation & GPU info
print("✅ Ultralytics ready")

Ultralytics 8.4.14  Python-3.12.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Setup complete  (16 CPUs, 31.6 GB RAM, 407.8/426.0 GB disk)
✅ Ultralytics ready


In [ ]:
# Step 0.1
import torch
 
print("="*50)
print("  HARDWARE CHECK")
print("="*50)
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")
 
if torch.cuda.is_available():
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = 0        # use GPU
else:
    print("⚠️ No GPU — will use CPU")
    DEVICE = 'cpu'
 
print(f"\n  Training device : {DEVICE}")

  HARDWARE CHECK
  PyTorch version : 2.5.1+cu121
  CUDA available  : True
  GPU             : NVIDIA GeForce RTX 4050 Laptop GPU
  VRAM            : 6.4 GB

  Training device : 0


In [ ]:
# Step 1.0
from ultralytics import YOLO
 
MODEL_SIZE = 'yolov8n-cls.pt'
 
model = YOLO(MODEL_SIZE)
print(f"✅ Loaded: {MODEL_SIZE}")
print(f"   Model parameters: {sum(p.numel() for p in model.model.parameters()):,}")
 

✅ Loaded: yolov8n-cls.pt
   Model parameters: 2,719,288


In [ ]:
# Step 1.1
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print(f"VRAM total: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")

VRAM free: 5.3 GB
VRAM total: 6.4 GB


In [ ]:
# Step 2.0
DATASET_YAML = "C:/Users/AHMAD ADAM HAKIMI/Downloads/Machine_Learning_Project/PlantVillage_Balanced"   # Folder that for new dataset augmentation
EPOCHS       = 50     
BATCH_SIZE   = 8     
IMG_SIZE     = 640
LR           = 0.01  
 
print("🚀 Starting YOLOv8 training...")
print(f"   Model    : {MODEL_SIZE}")
print(f"   Epochs   : {EPOCHS}")
print(f"   Batch    : {BATCH_SIZE}")
print(f"   Img size : {IMG_SIZE}")
print(f"   Device   : {DEVICE}")
print("-"*50)
 
results = model.train(
    data    = DATASET_YAML,   
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    device  = DEVICE,
    lr0     = LR,              
    lrf     = 0.01,            
    momentum= 0.937,          
    weight_decay = 0.0005,     
    warmup_epochs= 3,          
    patience = 15,             
    save     = True,           
    plots    = True,           
    verbose  = True,
    project  = "tomato_yolov8", 
    name     = "yolov8n", 
)
 
print("\n✅ Training complete!")
print(f"   Best model saved at: tomato_yolov8/yolov8n/weights/best.pt") 
 

🚀 Starting YOLOv8 training...
   Model    : yolov8n-cls.pt
   Epochs   : 50
   Batch    : 8
   Img size : 640
   Device   : 0
--------------------------------------------------
New https://pypi.org/project/ultralytics/8.4.60 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.12.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:/Users/AHMAD ADAM HAKIMI/Downloads/Machine_Learning_Project/PlantVillage_Balanced, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01

In [ ]:
# Step 3.0
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
 
run_dir = "runs/classify/tomato_yolov8/yolov8n"
 
# Ultralytics auto-save results.png
results_img = os.path.join(run_dir, "results.png")
 
if os.path.exists(results_img):
    img = mpimg.imread(results_img)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('YOLOv8 Training Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('runs/classify/tomato_yolov8/yolov8n/yolov8_train_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Training curves displayed")
else:
    print("⚠️ results.png not found — check folder:", run_dir)

<Figure size 1600x800 with 1 Axes>

✅ Training curves displayed


In [ ]:
# Step 4.0
from ultralytics import YOLO
import json
 
# Load best pre-trained model 
best_model = YOLO("runs/classify/tomato_yolov8/yolov8n/weights/best.pt")
 
print("📊 Evaluating on TEST set...")
test_results = best_model.val(
    data   = DATASET_YAML,
    split  = 'test',
    imgsz  = IMG_SIZE,
    device = 'cpu',
    plots  = True,
    project= "tomato_yolov8",
    name   = "yolov8n_test",
)
 
# Print key metrics
print("\n" + "="*50)
print("  TEST SET RESULTS — YOLOv8")
print("="*50)
print(f"  Top-1 Accuracy : {test_results.results_dict.get('metrics/accuracy_top1', 0)*100:.2f}%")
print(f"  Top-5 Accuracy : {test_results.results_dict.get('metrics/accuracy_top5', 0)*100:.2f}%")
print("="*50)

📊 Evaluating on TEST set...
Ultralytics 8.4.14  Python-3.12.9 torch-2.5.1+cu121 CPU (13th Gen Intel Core i7-13620H)
YOLOv8n-cls summary (fused): 30 layers, 1,447,690 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\AHMAD ADAM HAKIMI\Downloads\Machine_Learning_Project\PlantVillage_Balanced\train... found 44812 images in 10 classes  
val: C:\Users\AHMAD ADAM HAKIMI\Downloads\Machine_Learning_Project\PlantVillage_Balanced\val... found 2397 images in 10 classes  
test: C:\Users\AHMAD ADAM HAKIMI\Downloads\Machine_Learning_Project\PlantVillage_Balanced\test... found 2411 images in 10 classes  
test: Fast image access  (ping: 0.10.0 ms, read: 336.4110.2 MB/s, size: 101.8 KB)
test: Scanning C:\Users\AHMAD ADAM HAKIMI\Downloads\Machine_Learning_Project\PlantVillage_Balanced\test... 2411 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2411/2411  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 151/151 2.2it/s 1:080.4sss
                   all      0.995          1
Speed: 0.0ms prepr

In [ ]:
# Step 5.0
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path
import os
 
# Load best model
best_model = YOLO("runs/classify/tomato_yolov8/yolov8n/weights/best.pt")
 
# Collect predictions on test set
test_dir = "C:/Users/AHMAD ADAM HAKIMI/Downloads/Machine_Learning_Project/PlantVillage_Balanced/test"
y_true, y_pred = [], []
 
class_names_sorted = sorted(os.listdir(test_dir))
 
print("🔍 Running predictions on test images...")
for label_idx, cls in enumerate(class_names_sorted):
    cls_path = os.path.join(test_dir, cls)
    img_files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))]
 
    for img_file in img_files:
        img_path = os.path.join(cls_path, img_file)
        pred = best_model.predict(img_path, verbose=False, device='cpu')
        pred_class = pred[0].probs.top1   # index of top prediction
        y_true.append(label_idx)
        y_pred.append(pred_class)
 
# Short class labels for display
short_names = [c.replace('Tomato_','').replace('Tomato__','').replace('_',' ')
               for c in class_names_sorted]
 
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names,
            linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix — YOLOv8 (Test Set)', fontsize=13, fontweight='bold')
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('runs/classify/tomato_yolov8/yolov8n/yolov8_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: yolov8_confusion_matrix.png")
 
# --- Classification Report ---
print("\n📋 Classification Report — YOLOv8")
print("="*70)
report = classification_report(y_true, y_pred, target_names=short_names, digits=4)
print(report)
 
# Save report as text
with open("runs/classify/tomato_yolov8/yolov8n/yolov8_classification_report.txt", "w") as f:
    f.write("Classification Report — YOLOv8\n")
    f.write("="*70 + "\n")
    f.write(report)
print("✅ Saved: yolov8_classification_report.txt")

🔍 Running predictions on test images...


<Figure size 1400x1100 with 2 Axes>

✅ Saved: yolov8_confusion_matrix.png

📋 Classification Report — YOLOv8
                                      precision    recall  f1-score   support

                      Bacterial spot     0.9938    0.9969    0.9953       320
                        Early blight     0.9864    0.9667    0.9764       150
                         Late blight     0.9965    0.9826    0.9895       287
                           Leaf Mold     1.0000    1.0000    1.0000       144
                  Septoria leaf spot     0.9816    1.0000    0.9907       267
Spider mites Two spotted spider mite     1.0000    1.0000    1.0000       252
                         Target Spot     0.9953    1.0000    0.9976       212
              YellowLeaf  Curl Virus     1.0000    0.9979    0.9990       482
                        mosaic virus     1.0000    1.0000    1.0000        57
                             healthy     0.9959    1.0000    0.9979       240

                            accuracy                         0.9950  